# GLM-OCR checkpoint 642: capped-crop resolution validation

Inference-only experiment. It keeps checkpoint 642, the prompt, greedy decoding, stopping tokens, and the fixed 400-image validation set unchanged. Only validation crops capped by the current 1536-token spatial budget are regenerated at 2048 visual tokens. Saved 1536-token predictions are reused everywhere else, producing a controlled full-validation hybrid comparison. No training is performed.

In [ ]:
from __future__ import annotations
import gc, hashlib, json, math, platform, random, time, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch, transformers, peft
from IPython.display import display
from PIL import Image, ImageOps
from peft import PeftModel
from rapidfuzz.distance import Levenshtein
from tqdm.auto import tqdm
from transformers import AutoProcessor, GlmOcrForConditionalGeneration

ROOT = Path.cwd().resolve()
assert (ROOT / 'resources/Train.csv').is_file(), 'Open this notebook from the project root.'
RUN = ROOT / 'training_outputs/glm_ocr_from_base_curriculum_v7_native_ocr_balanced'
BASE_MODEL = ROOT / 'model/glm-ocr'
IMAGES = ROOT / 'resources/images'
CHECKPOINT = RUN / 'stage2/checkpoints/checkpoint-642'
BASELINE_PATH = RUN / 'stage2/validation/step-642_predictions.csv'
OUTPUT = RUN / 'higher_resolution_validation_checkpoint642'
CURRENT_VISUAL_TOKENS = 1536
HIGHER_VISUAL_TOKEN_BUDGETS = (2048,)
PIXELS_PER_VISUAL_TOKEN = 28 * 28
MIN_SPATIAL_PIXELS = 384 * PIXELS_PER_VISUAL_TOKEN
MAX_NEW_TOKENS = 128
BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 20260913

manifest = json.loads((RUN / 'run_identity.json').read_text(encoding='utf-8'))
CONFIG = manifest['identity']['config']
selection = json.loads((RUN / 'final_selection.json').read_text(encoding='utf-8'))
assert selection['spec'] == {'kind':'adapter','path':'stage2/checkpoints/checkpoint-642'}
assert CONFIG['max_spatial_pixels'] == CURRENT_VISUAL_TOKENS * PIXELS_PER_VISUAL_TOKEN
assert CHECKPOINT.joinpath('adapter_model.safetensors').is_file() and BASELINE_PATH.is_file()
assert torch.cuda.is_available(), 'A CUDA GPU is required for this inference-only experiment.'
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
DEVICE = 'cuda:0'

def normalize(value):
    return unicodedata.normalize('NFC', str(value)).strip()

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')
    temporary.replace(path)

validation = pd.read_csv(RUN / 'validation_manifest.csv', dtype={'ID':str}, keep_default_na=False).sort_values('ID').reset_index(drop=True)
baseline = pd.read_csv(BASELINE_PATH, dtype={'ID':str}, keep_default_na=False)[['ID','Target','Prediction']]
assert len(validation) == 400 and validation.ID.is_unique and baseline.ID.is_unique
assert set(validation.ID) == set(baseline.ID)
expected_targets = validation.set_index('ID').Target.map(normalize).sort_index()
saved_targets = baseline.set_index('ID').Target.map(normalize).sort_index()
assert expected_targets.equals(saved_targets), 'Saved baseline targets do not match the fixed validation manifest.'
validation['original_pixels'] = validation.width.astype(int) * validation.height.astype(int)
validation['current_capped'] = validation.original_pixels > CURRENT_VISUAL_TOKENS * PIXELS_PER_VISUAL_TOKEN
validation['tall'] = validation.height.astype(int) > 200
capped = validation.loc[validation.current_capped].copy().reset_index(drop=True)
assert len(capped) > 0 and capped.tall.all(), 'Expected every capped validation crop to be in the tall group.'
print('GPU:', torch.cuda.get_device_name(0))
print(f'Validation: {len(validation)} rows; tall: {int(validation.tall.sum())}; capped and rerun: {len(capped)}')
display(capped[['ID','width','height','original_pixels']].head())

In [ ]:
processor = AutoProcessor.from_pretrained(BASE_MODEL, local_files_only=True, use_fast=False)
base_generation = json.loads((BASE_MODEL / 'generation_config.json').read_text(encoding='utf-8'))
base_eos = base_generation['eos_token_id']
base_eos = [base_eos] if isinstance(base_eos, int) else list(base_eos)
EOS_IDS = sorted(set(base_eos + [processor.tokenizer.eos_token_id]))
PROMPT_TEXT = ('[gMASK]<sop><|user|>\n'
               '<|begin_of_image|><|image|><|end_of_image|>'
               + CONFIG['prompt'] + '<|assistant|>')

def resize_for_budget(image, visual_tokens):
    image = image.convert('RGB')
    maximum = int(visual_tokens) * PIXELS_PER_VISUAL_TOKEN
    for bound, comparison in [(maximum, lambda area, limit: area > limit),
                              (MIN_SPATIAL_PIXELS, lambda area, limit: area < limit)]:
        if comparison(image.width * image.height, bound):
            scale = math.sqrt(bound / (image.width * image.height))
            image = image.resize((max(1, int(image.width * scale)), max(1, int(image.height * scale))),
                                 Image.Resampling.BICUBIC)
    return image

def encode_image(image_id, visual_tokens):
    with Image.open(IMAGES / f'{image_id}.jpg') as source:
        image = resize_for_budget(ImageOps.exif_transpose(source).convert('RGB'), visual_tokens)
    inputs = processor(text=[PROMPT_TEXT], images=[image], add_special_tokens=False, return_tensors='pt')
    inputs.pop('token_type_ids', None)
    assert int(inputs['input_ids'][0,-1]) == processor.tokenizer.convert_tokens_to_ids('<|assistant|>')
    assert 'mm_token_type_ids' in inputs
    return inputs, image.size

def load_model():
    gc.collect(); torch.cuda.empty_cache()
    base = GlmOcrForConditionalGeneration.from_pretrained(
        BASE_MODEL, local_files_only=True, dtype=DTYPE, device_map={'':0}, attn_implementation='sdpa')
    model = PeftModel.from_pretrained(base, CHECKPOINT, is_trainable=False).eval()
    for parameter in model.parameters(): parameter.requires_grad_(False)
    return model

def predict(model, image_id, visual_tokens):
    inputs, spatial_size = encode_image(image_id, visual_tokens)
    visual_token_count = int(inputs['image_grid_thw'].prod().item() // 4)
    inputs = inputs.to(DEVICE)
    torch.cuda.synchronize(); started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=1, do_sample=False,
                                use_cache=True, eos_token_id=EOS_IDS,
                                pad_token_id=processor.tokenizer.pad_token_id)
    torch.cuda.synchronize(); elapsed = time.perf_counter() - started
    generated = output[0, inputs['input_ids'].shape[1]:]
    prediction = normalize(processor.decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=False))
    hit_limit = len(generated) >= MAX_NEW_TOKENS and int(generated[-1]) not in EOS_IDS
    return {'Prediction':prediction, 'visual_tokens':visual_token_count,
            'resized_width':spatial_size[0], 'resized_height':spatial_size[1],
            'generated_tokens':len(generated), 'hit_token_limit':bool(hit_limit), 'seconds':elapsed}

In [ ]:
# This is the only GPU loop: 89 capped validation crops, no training. Results are resumably cached.
experiment_identity = {
    'checkpoint':'stage2/checkpoints/checkpoint-642',
    'checkpoint_weights':file_hash(CHECKPOINT / 'adapter_model.safetensors'),
    'baseline_predictions':file_hash(BASELINE_PATH),
    'validation_manifest':file_hash(RUN / 'validation_manifest.csv'),
    'capped_ids':capped.ID.tolist(), 'current_visual_tokens':CURRENT_VISUAL_TOKENS,
    'higher_visual_token_budgets':list(HIGHER_VISUAL_TOKEN_BUDGETS),
    'prompt':CONFIG['prompt'], 'max_new_tokens':MAX_NEW_TOKENS, 'num_beams':1,
    'runtime':{'python':platform.python_version(), 'torch':torch.__version__,
               'transformers':transformers.__version__, 'peft':peft.__version__},
}
EXPERIMENT_KEY = object_hash(experiment_identity)
OUT = OUTPUT / EXPERIMENT_KEY[:16]
OUT.mkdir(parents=True, exist_ok=True)
identity_path = OUT / 'experiment.json'
if identity_path.exists():
    assert json.loads(identity_path.read_text(encoding='utf-8')) == experiment_identity
else:
    write_json(identity_path, experiment_identity)

def cache_path(visual_tokens, image_id):
    return OUT / 'cache' / f'visual_tokens_{visual_tokens}' / f'{image_id}.json'

def cached_prediction(visual_tokens, image_id):
    path = cache_path(visual_tokens, image_id)
    if not path.exists(): return None
    row = json.loads(path.read_text(encoding='utf-8'))
    assert row['experiment_key'] == EXPERIMENT_KEY and row['ID'] == image_id
    assert row['visual_token_budget'] == visual_tokens
    return row

pending = [(budget, image_id) for budget in HIGHER_VISUAL_TOKEN_BUDGETS for image_id in capped.ID
           if cached_prediction(budget, image_id) is None]
print(f'Experiment: {OUT}; cached: {len(capped)*len(HIGHER_VISUAL_TOKEN_BUDGETS)-len(pending)}/'
      f'{len(capped)*len(HIGHER_VISUAL_TOKEN_BUDGETS)}')
if pending:
    model = load_model()
    try:
        warm_inputs, _ = encode_image(pending[0][1], pending[0][0])
        with torch.inference_mode():
            model.generate(**warm_inputs.to(DEVICE), max_new_tokens=2, num_beams=1, do_sample=False)
        del warm_inputs
        for budget in HIGHER_VISUAL_TOKEN_BUDGETS:
            for image_id in tqdm(capped.ID, desc=f'capped crops @ {budget}'):
                if cached_prediction(budget, image_id) is not None: continue
                result = predict(model, image_id, budget)
                write_json(cache_path(budget, image_id), {
                    'experiment_key':EXPERIMENT_KEY, 'ID':image_id,
                    'visual_token_budget':budget, **result})
    finally:
        del model; gc.collect(); torch.cuda.empty_cache()
print('Capped-crop inference complete.')

In [ ]:
def add_errors(frame):
    frame = frame.copy()
    frame['Target'] = frame.Target.map(normalize); frame['Prediction'] = frame.Prediction.map(normalize)
    frame['nwords'] = frame.Target.str.split().str.len().clip(lower=1)
    frame['nchars'] = frame.Target.str.len().clip(lower=1)
    frame['word_edits'] = [Levenshtein.distance(a.split(), b.split()) for a,b in zip(frame.Target,frame.Prediction)]
    frame['char_edits'] = [Levenshtein.distance(a,b) for a,b in zip(frame.Target,frame.Prediction)]
    return frame

def summarize(frame):
    frame = add_errors(frame) if 'word_edits' not in frame else frame
    word_weight = np.sqrt(frame.nwords); char_weight = np.sqrt(frame.nchars)
    weighted_wer = np.dot(frame.word_edits, word_weight) / word_weight.sum()
    weighted_cer = np.dot(frame.char_edits, char_weight) / char_weight.sum()
    return {'rows':len(frame), 'weighted_wer':float(weighted_wer), 'weighted_cer':float(weighted_cer),
            'zindi_score_proxy':float(1-.5*(weighted_wer/CONFIG['word_normalizer']
                                                +weighted_cer/CONFIG['character_normalizer'])),
            'WER_percent':float(100*frame.word_edits.sum()/frame.nwords.sum()),
            'CER_percent':float(100*frame.char_edits.sum()/frame.nchars.sum()),
            'exact_lines':int(frame.Target.eq(frame.Prediction).sum())}

baseline_full = validation[['ID','Target','current_capped','tall']].merge(
    baseline[['ID','Prediction']], on='ID', validate='one_to_one')
summary_rows = [{'candidate':'baseline_1536', 'visual_token_budget':CURRENT_VISUAL_TOKENS,
                 'scope':'full_validation', **summarize(baseline_full)}]
comparison_frames = {}
for budget in HIGHER_VISUAL_TOKEN_BUDGETS:
    generated = pd.DataFrame([cached_prediction(budget, image_id) for image_id in capped.ID])
    assert len(generated) == len(capped) and generated.Prediction.map(normalize).ne('').all()
    hybrid = baseline_full.copy().rename(columns={'Prediction':'BaselinePrediction'})
    hybrid = hybrid.merge(generated[['ID','Prediction']].rename(columns={'Prediction':'HigherPrediction'}),
                          on='ID', how='left', validate='one_to_one')
    hybrid['Prediction'] = hybrid.HigherPrediction.fillna(hybrid.BaselinePrediction)
    higher_errors = add_errors(hybrid[['ID','Target','Prediction','current_capped','tall']])
    baseline_errors = add_errors(hybrid[['ID','Target','BaselinePrediction','current_capped','tall']].rename(
        columns={'BaselinePrediction':'Prediction'}))
    hybrid['baseline_word_edits'] = baseline_errors.word_edits
    hybrid['higher_word_edits'] = higher_errors.word_edits
    hybrid['baseline_char_edits'] = baseline_errors.char_edits
    hybrid['higher_char_edits'] = higher_errors.char_edits
    hybrid['result'] = np.select(
        [(hybrid.higher_word_edits < hybrid.baseline_word_edits) | (hybrid.higher_char_edits < hybrid.baseline_char_edits),
         (hybrid.higher_word_edits > hybrid.baseline_word_edits) | (hybrid.higher_char_edits > hybrid.baseline_char_edits)],
        ['improved','worsened'], default='tied')
    comparison_frames[budget] = hybrid
    for scope, mask in [('capped_only',hybrid.current_capped), ('tall_all',hybrid.tall),
                        ('full_validation',pd.Series(True,index=hybrid.index))]:
        candidate_metrics = summarize(higher_errors.loc[mask])
        baseline_metrics = summarize(baseline_errors.loc[mask])
        summary_rows.append({'candidate':f'hybrid_{budget}', 'visual_token_budget':budget, 'scope':scope,
                             **candidate_metrics,
                             'score_before':baseline_metrics['zindi_score_proxy'],
                             'score_delta':candidate_metrics['zindi_score_proxy']-baseline_metrics['zindi_score_proxy']})
    hybrid.to_csv(OUT / f'row_comparison_{budget}.csv', index=False)

summary = pd.DataFrame(summary_rows)
baseline_score = float(summary.loc[(summary.candidate=='baseline_1536') &
                                   (summary.scope=='full_validation'),'zindi_score_proxy'].iloc[0])
summary.loc[summary.candidate.eq('baseline_1536'),'score_before'] = baseline_score
summary.loc[summary.candidate.eq('baseline_1536'),'score_delta'] = 0.0
summary.to_csv(OUT / 'resolution_summary.csv', index=False)
best_row = summary.loc[summary.scope.eq('full_validation')].sort_values('zindi_score_proxy',ascending=False).iloc[0]
write_json(OUT / 'result.json', {'baseline_full_score':baseline_score,
    'best_candidate':best_row.candidate, 'best_full_score':float(best_row.zindi_score_proxy),
    'best_full_score_delta':float(best_row.zindi_score_proxy-baseline_score),
    'capped_rows':len(capped), 'tall_rows':int(validation.tall.sum()),
    'recommendation':'use_higher_resolution_for_capped_crops' if best_row.candidate!='baseline_1536' else 'keep_1536'})
display(summary.sort_values(['scope','zindi_score_proxy'],ascending=[True,False]))
for budget, frame in comparison_frames.items():
    print(f'{budget} capped outcomes:', frame.loc[frame.current_capped,'result'].value_counts().to_dict())
print(json.dumps(json.loads((OUT/'result.json').read_text()), indent=2))
print('Saved reports:', OUT)